In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../data")
print("Files in data/:", [f.name for f in DATA.iterdir()])


Files in data/: ['bodmas_malware_category.csv', 'bodmas.npz', 'bodmas_metadata.csv', 'test_pe']


In [10]:
meta = pd.read_csv(DATA / "bodmas_metadata.csv")
print("shape:", meta.shape)
print("columns:", list(meta.columns))
meta.head()

shape: (134435, 3)
columns: ['sha', 'timestamp', 'family']


,sha,timestamp,family
0,e6d7b4bab32def853ab564410df53fa33172dda1bfd48c...,2007-01-01 08:46:39+00:00,NaN
1,5af37a058a5bcf2284c183ee98d92b7c66d8f5ce623e92...,2007-01-26 17:16:30+00:00,NaN
2,5bfbbea150af5cef2d3a93b80ef7c7faea9f564b56045d...,2007-03-21 02:08:53+00:00,NaN
3,216f592f1e1717d5681b7f5f2b14a28a2f0c603b5b7318...,2007-04-25 12:55:06+00:00,NaN
4,a1ca76813d2e9e7e23b830c87fbe29bcb51fcbe096e445...,2007-11-14 15:03:55+00:00,NaN


In [11]:
# How many benign vs malware (malware = has a family label)
print("Total rows:", len(meta))
print("Benign (no family):", meta["family"].isna().sum())
print("Malware (has family):", meta["family"].notna().sum())
print("Number of distinct families:", meta["family"].nunique())



Total rows: 134435
Benign (no family): 77142
Malware (has family): 57293
Number of distinct families: 582


In [12]:
cat = pd.read_csv(DATA / "bodmas_malware_category.csv")
print("shape:", cat.shape)
print("columns:", list(cat.columns))
cat.head(10)

shape: (57293, 2)
columns: ['sha256', 'category']


,sha256,category
0,6a695877f571d043fe08d3cc715d9d4b4af85ffe837fa0...,worm
1,9ef9439795cac85e711b59df296a19e7ac43c144035f2f...,trojan
2,32de655f9010d8d152db16c6e5bbad215fa09286a08ff1...,worm
3,a68f7fb26ad84859625002395cf67f22ea0956996ed9c8...,downloader
4,d5c74472adfda20166a65f8b2886819a014ebcb67b999e...,trojan
5,c9c7384b5cd0bf22f6366caeb52af5e8f3d1265cbbb9d1...,worm
6,774b0ff235a2f50d983ca6b1fd6878a98bd3659954b63e...,trojan
7,b9198895114cea953666dc9380e55366759f9b769f1b3a...,worm
8,02450f54a5513dac224337207c8d6380d28e7e90f51c6d...,worm
9,7d8ea61dc1a7a6f84087d49a5bf0503925b07fbd8c2252...,worm


In [13]:
print("Distinct categories:", cat["category"].nunique())
print()
print(cat["category"].value_counts())


Distinct categories: 14

category
trojan                29972
worm                  16697
backdoor               7331
downloader             1031
ransomware              821
dropper                 715
informationstealer      448
virus                   192
pua                      29
cryptominer              20
p2p-worm                 16
exploit                  12
trojan-gamethief          6
rootkit                   3
Name: count, dtype: int64


In [14]:
import lief

pe = lief.parse("../data/test_pe/putty.exe")

print("=== Parsed:", pe.name if hasattr(pe, "name") else "putty.exe", "===")
print("\n--- Sections (name, size, entropy) ---")
for s in pe.sections:
    print(f"{s.name:10s}  size={s.size:>8}  entropy={s.entropy:.2f}")

=== Parsed: putty.exe ===

--- Sections (name, size, entropy) ---
.text       size=  970752  entropy=6.45
.rdata      size=  281600  entropy=5.41
.data       size=    4096  entropy=2.08
.pdata      size=   29184  entropy=5.91
.00cfg      size=     512  entropy=2.71
.gxfg       size=   11264  entropy=5.25
.tls        size=     512  entropy=-0.00
_RDATA      size=     512  entropy=4.13
.rsrc       size=  374272  entropy=7.83
.reloc      size=    8704  entropy=5.44


In [15]:
print("--- Imported libraries and a few functions each ---")
if pe.has_imports:
    for lib in pe.imports:
        funcs = [e.name for e in lib.entries if e.name]
        print(f"\n{lib.name}  ({len(funcs)} functions)")
        print("   ", ", ".join(funcs[:8]), "..." if len(funcs) > 8 else "")
else:
    print("No imports found")

--- Imported libraries and a few functions each ---

GDI32.dll  (51 functions)
    BitBlt, CreateBitmap, CreateCompatibleBitmap, CreateCompatibleDC, CreateFontA, CreateFontIndirectA, CreatePalette, CreatePen ...

IMM32.dll  (5 functions)
    ImmGetCompositionStringW, ImmGetContext, ImmReleaseContext, ImmSetCompositionFontA, ImmSetCompositionWindow 

ole32.dll  (3 functions)
    CoCreateInstance, CoInitialize, CoUninitialize 

USER32.dll  (119 functions)
    AppendMenuA, BeginPaint, CheckDlgButton, CheckMenuItem, CheckRadioButton, CloseClipboard, CreateCaret, CreateDialogParamA ...

KERNEL32.dll  (148 functions)
    Beep, ClearCommBreak, CloseHandle, CompareStringW, ConnectNamedPipe, CreateEventA, CreateFileA, CreateFileMappingA ...

SHELL32.dll  (1 functions)
    ShellExecuteA 

COMDLG32.dll  (6 functions)
    ChooseColorA, ChooseFontA, GetOpenFileNameA, GetOpenFileNameW, GetSaveFileNameA, GetSaveFileNameW 

ADVAPI32.dll  (15 functions)
    AllocateAndInitializeSid, CopySid, EqualSid, 

In [16]:
data = np.load(DATA / "bodmas.npz")
X = data["X"]
y = data["y"]

print("X shape:", X.shape, " y shape:", y.shape)
print("X dtype:", X.dtype)
print("first 10 values of row 0:", X[0, :10])

X shape: (134435, 2381)  y shape: (134435,)
X dtype: float32
first 10 values of row 0: [0.05674198 0.00801749 0.00776239 0.00546647 0.00776239 0.00444606
 0.00543003 0.00306122 0.00947522 0.00630466]


In [17]:
import pandas as pd
from pathlib import Path

DATA = Path("../data")

# load the category labels (sha256 -> category)
cat = pd.read_csv(DATA / "bodmas_malware_category.csv")

# the 8 categories we care about
KEEP = ["trojan", "worm", "backdoor", "downloader",
        "ransomware", "informationstealer", "dropper", "virus"]

# pick a few samples from each category so the PoC spans classes
picks = []
for c in KEEP:
    rows = cat[cat["category"] == c].head(3)   # first 3 of each
    picks.append(rows)

sample = pd.concat(picks).reset_index(drop=True)
print("total picked:", len(sample))
print(sample["category"].value_counts())
sample

total picked: 24
category
trojan                3
worm                  3
backdoor              3
downloader            3
ransomware            3
informationstealer    3
dropper               3
virus                 3
Name: count, dtype: int64


,sha256,category
0,9ef9439795cac85e711b59df296a19e7ac43c144035f2f...,trojan
1,d5c74472adfda20166a65f8b2886819a014ebcb67b999e...,trojan
2,774b0ff235a2f50d983ca6b1fd6878a98bd3659954b63e...,trojan
3,6a695877f571d043fe08d3cc715d9d4b4af85ffe837fa0...,worm
4,32de655f9010d8d152db16c6e5bbad215fa09286a08ff1...,worm
5,c9c7384b5cd0bf22f6366caeb52af5e8f3d1265cbbb9d1...,worm
6,b04a8f775aae9cc1eeb35deb125432fffd385c1f71d317...,backdoor
7,148200ba66e55055a1a1423e44bde143e7d7a550122f7e...,backdoor
8,c5b03ce62adbf84fb26bd00e5826fabdc13224d01f965d...,backdoor
9,a68f7fb26ad84859625002395cf67f22ea0956996ed9c8...,downloader


In [18]:
import zipfile

ZIP_PATH = "/Volumes/Bodmas/BODMAS_disarmed_malware_binaries.zip"
OUT_DIR = Path("/Volumes/Bodmas/binaries")

with zipfile.ZipFile(ZIP_PATH) as z:
    for sha in sample["sha256"]:
        member = f"altered/{sha}.exe"          # its name inside the zip
        target = OUT_DIR / f"{sha}.exe"         # where it lands on the SSD
        with z.open(member) as src, open(target, "wb") as dst:
            dst.write(src.read())

print("extracted", len(sample), "files")
print("on disk:", len(list(OUT_DIR.glob("*.exe"))), "exe files")

extracted 24 files
on disk: 24 exe files


In [21]:
import sys
sys.path.append("../src")          # so we can import from the src folder
from extract import extract_features

# grab the first extracted binary
BIN_DIR = Path("/Volumes/Bodmas/binaries")
first = sorted(BIN_DIR.glob("*.exe"))[0]
print("testing on:", first.name)

result = extract_features(str(first))

if result is None:
    print("FAILED: LIEF returned None — the disarmed header broke parsing")
else:
    print("SUCCESS")
    print("num_sections:", result["num_sections"])
    print("section_names:", result["section_names"])
    print("num DLLs imported:", len(result["imports"]))

testing on: 0a19f43b9afac2da5685988dd4a494096ac271b7a8038481de3fc8c09a16da8d.exe
SUCCESS
num_sections: 9
section_names: ['CODE', 'DATA', 'BSS', '.idata', '.tls', '.rdata', '.reloc', '.rsrc', '.aspack']
num DLLs imported: 10


Failed to parse COFF string table
Failed to parse COFF symbol #0


In [22]:
label_map = dict(zip(cat["sha256"], cat["category"]))

rows = []
skipped = []

for path in sorted(BIN_DIR.glob("*.exe")):
    sha = path.stem
    record = extract_features(str(path))

    if record is None:
        skipped.append(sha)
        continue

    record["sha256"] = sha
    record["category"] = label_map[sha]
    rows.append(record)

features_df = pd.DataFrame(rows)

print("extracted:", len(features_df), "  skipped:", len(skipped))
print(features_df["category"].value_counts())
features_df.head()

Failed to parse COFF string table
Failed to parse COFF symbol #0
Failed to parse COFF string table
Failed to parse COFF symbol #0
Address of new exe header is corrupted
Failed to parse DOS Stub
Address of new exe header is corrupted
Failed to parse DOS Stub
Address of new exe header is corrupted
Failed to parse DOS Stub
Address of new exe header is corrupted
Failed to parse DOS Stub


extracted: 24   skipped: 0
category
virus                 3
ransomware            3
backdoor              3
informationstealer    3
worm                  3
dropper               3
trojan                3
downloader            3
Name: count, dtype: int64


,path,num_sections,section_names,section_entropies,imports,sha256,category
0,/Volumes/Bodmas/binaries/0a19f43b9afac2da56859...,9,"[CODE, DATA, BSS, .idata, .tls, .rdata, .reloc...","{'CODE': 6.52, 'DATA': 2.8, 'BSS': -0.0, '.ida...","{'kernel32.dll': ['Sleep'], 'user32.dll': ['Cr...",0a19f43b9afac2da5685988dd4a494096ac271b7a80384...,virus
1,/Volumes/Bodmas/binaries/13f35ef23b6c9aa653347...,6,"[.text, .rdata, .data, .gfids, .rsrc, .reloc]","{'.text': 6.67, '.rdata': 5.57, '.data': 3.07,...","{'KERNEL32.dll': ['GetConsoleOutputCP', 'Devic...",13f35ef23b6c9aa653347b4fcfb5fc59fbef901dc5eace...,ransomware
2,/Volumes/Bodmas/binaries/148200ba66e55055a1a14...,8,"[CODE, DATA, BSS, .idata, .tls, .rdata, .reloc...","{'CODE': 6.58, 'DATA': 4.78, 'BSS': 0.0, '.ida...",{'kernel32.dll': ['WritePrivateProfileStringA'...,148200ba66e55055a1a1423e44bde143e7d7a550122f7e...,backdoor
3,/Volumes/Bodmas/binaries/16a0e621fe72144448862...,9,"[CODE, DATA, BSS, .idata, .tls, .rdata, .reloc...","{'CODE': 6.4, 'DATA': 5.44, 'BSS': 0.0, '.idat...","{'kernel32.dll': ['GetProcAddress', 'GetModule...",16a0e621fe72144448862ab550e3e94d565c5016aff677...,informationstealer
4,/Volumes/Bodmas/binaries/17e703d0d0ff76b93b8db...,3,"[.text, .rsrc, .reloc]","{'.text': 7.68, '.rsrc': 4.36, '.reloc': 1.95}",{'mscoree.dll': ['_CorExeMain']},17e703d0d0ff76b93b8db25ba0846a3fad076381747a17...,informationstealer


In [23]:
OUT = Path("../data/features_poc.jsonl")
features_df.to_json(OUT, orient="records", lines=True)
print("saved", len(features_df), "rows to", OUT)

check = pd.read_json(OUT, lines=True)
print("read back:", len(check), "rows")
print("columns:", list(check.columns))

saved 24 rows to ../data/features_poc.jsonl
read back: 24 rows
columns: ['path', 'num_sections', 'section_names', 'section_entropies', 'imports', 'sha256', 'category']


In [24]:
features_df[["sha256", "category", "num_sections", "section_names", "imports"]].head(5)

,sha256,category,num_sections,section_names,imports
0,0a19f43b9afac2da5685988dd4a494096ac271b7a80384...,virus,9,"[CODE, DATA, BSS, .idata, .tls, .rdata, .reloc...","{'kernel32.dll': ['Sleep'], 'user32.dll': ['Cr..."
1,13f35ef23b6c9aa653347b4fcfb5fc59fbef901dc5eace...,ransomware,6,"[.text, .rdata, .data, .gfids, .rsrc, .reloc]","{'KERNEL32.dll': ['GetConsoleOutputCP', 'Devic..."
2,148200ba66e55055a1a1423e44bde143e7d7a550122f7e...,backdoor,8,"[CODE, DATA, BSS, .idata, .tls, .rdata, .reloc...",{'kernel32.dll': ['WritePrivateProfileStringA'...
3,16a0e621fe72144448862ab550e3e94d565c5016aff677...,informationstealer,9,"[CODE, DATA, BSS, .idata, .tls, .rdata, .reloc...","{'kernel32.dll': ['GetProcAddress', 'GetModule..."
4,17e703d0d0ff76b93b8db25ba0846a3fad076381747a17...,informationstealer,3,"[.text, .rsrc, .reloc]",{'mscoree.dll': ['_CorExeMain']}


In [25]:
import sys
sys.path.append("../src")
from enrich import enrich_one

sample_row = features_df.iloc[0].to_dict()

print("CATEGORY (for your reference only):", sample_row["category"])
print("SHA:", sample_row["sha256"][:16])
print()
print("FEATURES GIVEN TO MODEL:")
print("  sections:", sample_row["section_names"])
print("  imports:", list(sample_row["imports"].keys()))
print()
print("=" * 60)
print("MISTRAL DESCRIPTION:")
print("=" * 60)
description = enrich_one(sample_row)
print(description)

CATEGORY (for your reference only): virus
SHA: 0a19f43b9afac2da

FEATURES GIVEN TO MODEL:
  sections: ['CODE', 'DATA', 'BSS', '.idata', '.tls', '.rdata', '.reloc', '.rsrc', '.aspack']
  imports: ['kernel32.dll', 'user32.dll', 'advapi32.dll', 'oleaut32.dll', 'version.dll', 'gdi32.dll', 'ole32.dll', 'comctl32.dll', 'shell32.dll', 'ADVAPI32.DLL']

MISTRAL DESCRIPTION:
The program appears to be a graphical user interface (GUI) application with window management capabilities, potentially using custom images for its UI. It can interact with system parameters, windows, menus, cursors, and clipboard data. It may also handle dialog boxes, pop-ups, scrollbars, and system fonts. Network activity is not evident from the provided features, nor are file operations or process/memory manipulation indicators. Evasion techniques or packing are not suggested by the given evidence.
